# P300 speller (calibration mode)

- Target letter is known each time, so every falsh can be labelled as target or non-target

## Packages

- source .speller/bin/activate
- deactivate

- conda activate speller
- conda install -c conda-forge psychopy -y

- python 3.9 but also 3.10 works
- psychopy - conda install -c conda-forge "pyglet<2" -y
- pylsl

## LSL + Faces

- sequence is 12 flashses total 
- 1 row, 1 column falsh is target
- oher 10 are non-target

Markers
* **`experiment_start` / `experiment_end`**: mark the beginning and end of the whole speller session
* **`focus_start/<char>` / `focus_end/<char>`**: mark the period where the participant is shown which character to focus on
* **`target_start/<char>` / `target_end/<char>`**: mark the start and end of one target-character block
* **`sequence_start/<char>/<seq>` / `sequence_end/<char>/<seq>`**: mark the start and end of one full shuffled row+column flashing sequence
* **`flash_on/<kind>/<idx>/target_<0 or 1>/char_<char>/seq_<seq>/flash_<flash>`**: marks the exact onset of a flashed row or column, used to time-lock EEG epochs for P300 analysis
* **`flash_off/<kind>/<idx>/target_<0 or 1>/char_<char>/seq_<seq>/flash_<flash>`**: marks the end of that flash, mainly useful for timing checks and debugging
* **`target_1` vs `target_0`**: indicates whether the flashed row/column contains the attended target character, giving the label for CNN training
* **`kind` and `idx`**: specify which row or column was flashed, so predictions can later be combined into the final selected character



Full pipeline needed: 
1. Speller
- sends marker stream via LSL
2. EEG acquisition software 
- sends EEG stream via LSL
3. Recorder
- records EEG + markers into XDF
4. Processing
- reads the EEG marker timing, epochs the data, trains / runs CNN

In [ ]:
from psychopy import visual, core, event
from PIL import Image
from pylsl import StreamInfo, StreamOutlet, local_clock
import random
import os


# Parameters ---------------------
GRID_ROWS = 6
GRID_COLS = 6

# Flash timing (seconds)
FLASH_DUR = 0.100   # how long a row/col stays highlighted
ISI       = 0.075   # pause between flashes
N_SEQS    = 10      # how many full row+col sequences per target letter
LETTER_CUE_DUR = 2   # time to show target letter before flashing
LETTER_BREAK_DUR = 2 # break between letters

# Window settings
BG_COLOR = "black"
UNITS = "pix"

# Grid layout
X_SPACING = 180
Y_SPACING = 130

# Cell box size
CELL_W = 120
CELL_H = 120

# Max displayed size of each image in pixels
IMG_MAX_W = 110
IMG_MAX_H = 110

# Text size
TEXT_HEIGHT = 36

# Colors
BOX_COLOR = "white"
HIGHLIGHT_COLOR = "yellow"
TEXT_COLOR = "white"

# Folder with your cell images
IMAGE_DIR = "Images"

# Word to spell during calibration
TARGET_WORD = "BRAIN"

# LSL marker stream name
MARKER_STREAM_NAME = "P300Markers"
MARKER_STREAM_TYPE = "Markers"

# 6x6 speller characters
GRID_SYMBOLS = [
    ["A", "B", "C", "D", "E", "F"],
    ["G", "H", "I", "J", "K", "L"],
    ["M", "N", "O", "P", "Q", "R"],
    ["S", "T", "U", "V", "W", "X"],
    ["Y", "Z", "1", "2", "3", "4"],
    ["5", "6", "7", "8", "9", "0"],
]


# Helper functions ---------------------
def check_escape():
    # Allow quitting anytime with Escape or Q
    keys = event.getKeys(keyList=["escape", "q"])
    return "escape" in keys or "q" in keys


def get_image_dir():
    folder = os.path.abspath(IMAGE_DIR)
    if os.path.isdir(folder):
        print("Using image folder:", folder)
        return folder
    raise FileNotFoundError(f"Could not find image folder: {folder}")


def make_image_stim(win, img_path, pos, max_w=IMG_MAX_W, max_h=IMG_MAX_H):
    # Open image to get original pixel size
    with Image.open(img_path) as pil_img:
        orig_w, orig_h = pil_img.size

    # Preserve aspect ratio and fit inside max box
    scale = min(max_w / orig_w, max_h / orig_h)
    disp_w = max(1, int(orig_w * scale))
    disp_h = max(1, int(orig_h * scale))

    # Create image stimulus
    image_stim = visual.ImageStim(
        win=win,
        image=img_path,
        pos=pos,
        size=(disp_w, disp_h),
        units=UNITS,
        interpolate=True
    )
    return image_stim


def build_grid_positions(rows, cols, x_spacing, y_spacing):
    # Build centered grid positions in pixel units
    positions = []

    total_w = (cols - 1) * x_spacing
    total_h = (rows - 1) * y_spacing

    start_x = -total_w / 2
    start_y = total_h / 2

    for r in range(rows):
        for c in range(cols):
            x = start_x + c * x_spacing
            y = start_y - r * y_spacing
            positions.append((r, c, (x, y)))

    return positions


def draw_message(win, text, y=0, height=28):
    # Draw text on screen
    msg = visual.TextStim(
        win=win,
        text=text,
        pos=(0, y),
        height=height,
        color=TEXT_COLOR,
        units=UNITS
    )
    msg.draw()


def wait_with_escape(duration):
    # Wait for a duration while still checking quit keys
    timer = core.Clock()
    while timer.getTime() < duration:
        if check_escape():
            return False
        core.wait(0.005)
    return True


def find_target_position(target_letter):
    # Find target letter row and column in the 6x6 grid
    for r in range(GRID_ROWS):
        for c in range(GRID_COLS):
            if GRID_SYMBOLS[r][c] == target_letter:
                return r, c
    return None, None


def draw_grid(cells, target_row=None, target_col=None, highlight_type=None, highlight_index=None, show_target_image_only=False):
    # Normal state:
    # draw letters only
    #
    # Target cue state:
    # draw only the target image in the target cell
    #
    # During row flash:
    # show images in flashed row, no letters in flashed row
    #
    # During column flash:
    # show images in flashed column, no letters in flashed column

    for cell in cells:
        row = cell["row"]
        col = cell["col"]

        is_target = (row == target_row and col == target_col)
        is_row_flash = (highlight_type == "row" and row == highlight_index)
        is_col_flash = (highlight_type == "col" and col == highlight_index)
        is_flashed = is_row_flash or is_col_flash

        if show_target_image_only and is_target:
            cell["box"].lineColor = HIGHLIGHT_COLOR
            cell["box"].lineWidth = 4
        elif is_flashed:
            cell["box"].lineColor = HIGHLIGHT_COLOR
            cell["box"].lineWidth = 4
        else:
            cell["box"].lineColor = BOX_COLOR
            cell["box"].lineWidth = 2

        cell["box"].draw()

        if show_target_image_only:
            if is_target:
                cell["image"].draw()
            else:
                cell["text"].draw()
        elif is_flashed:
            cell["image"].draw()
        else:
            cell["text"].draw()


def make_marker_outlet():
    info = StreamInfo(
        name=MARKER_STREAM_NAME,
        type=MARKER_STREAM_TYPE,
        channel_count=1,
        nominal_srate=0,
        channel_format="string",
        source_id="p300-psychopy-markers"
    )
    return StreamOutlet(info)


def send_marker(outlet, marker):
    outlet.push_sample([marker], local_clock())
    print("[MARKER]", marker)


def main():
    win = None
    outlet = None
    experiment_started = False

    try:
        image_dir_abs = get_image_dir()
        outlet = make_marker_outlet()

        # Window ---------------------
        win = visual.Window(
            fullscr=True,
            color=BG_COLOR,
            units=UNITS,
            checkTiming=False
        )

        win_w, win_h = win.size

        # Dynamic layout positions ---------------------
        title_y = win_h / 2 - 60
        bottom_y = -win_h / 2 + 50

        # Move grid slightly downward so title does not overlap
        grid_center_shift_y = -40

        # Load image paths ---------------------
        image_files = [f"cell_{i:02d}.jpg" for i in range(GRID_ROWS * GRID_COLS)]
        image_paths = [os.path.join(image_dir_abs, f) for f in image_files]

        # Check if all images exist
        missing = [p for p in image_paths if not os.path.exists(p)]
        if missing:
            print("Missing image files:")
            for m in missing:
                print(m)

            while True:
                draw_message(win, "Missing image files", y=40, height=30)
                draw_message(win, "Check terminal output for exact missing paths", y=0, height=22)
                draw_message(win, "Press ESC or Q to close", y=-40, height=24)
                win.flip()

                if check_escape():
                    return
                core.wait(0.01)

        # Build grid ---------------------
        cells = []
        positions = build_grid_positions(GRID_ROWS, GRID_COLS, X_SPACING, Y_SPACING)

        for idx, (row, col, pos) in enumerate(positions):
            symbol = GRID_SYMBOLS[row][col]
            shifted_pos = (pos[0], pos[1] + grid_center_shift_y)

            # Border box
            box_stim = visual.Rect(
                win=win,
                pos=shifted_pos,
                width=CELL_W,
                height=CELL_H,
                units=UNITS,
                lineColor=BOX_COLOR,
                fillColor=None,
                lineWidth=2
            )

            # Letter / number text
            text_stim = visual.TextStim(
                win=win,
                text=symbol,
                pos=shifted_pos,
                height=TEXT_HEIGHT,
                color=TEXT_COLOR,
                units=UNITS
            )

            # Image used during flashed row/column
            image_stim = make_image_stim(
                win=win,
                img_path=image_paths[idx],
                pos=shifted_pos,
                max_w=IMG_MAX_W,
                max_h=IMG_MAX_H
            )

            cells.append({
                "index": idx,
                "row": row,
                "col": col,
                "symbol": symbol,
                "pos": shifted_pos,
                "box": box_stim,
                "text": text_stim,
                "image": image_stim
            })

        running = True

        # Instructions screen ---------------------
        while running:
            if check_escape():
                running = False
                break

            draw_message(win, "Calibration Speller", y=title_y, height=34)
            draw_grid(cells)
            draw_message(win, "Press SPACE to start, ESC or Q to quit", y=bottom_y, height=24)
            win.flip()

            event.clearEvents(eventType="keyboard")
            keys = event.waitKeys(keyList=["space", "escape", "q"])

            if keys:
                if "space" in keys:
                    send_marker(outlet, "experiment_start")
                    experiment_started = True
                    break
                if "escape" in keys or "q" in keys:
                    running = False
                    break

        # Calibration loop ---------------------
        if running:
            for letter_idx, target_letter in enumerate(TARGET_WORD):
                target_row, target_col = find_target_position(target_letter)

                send_marker(outlet, f"target_start/{target_letter}")
                send_marker(outlet, f"focus_start/{target_letter}")

                # Show target cue before flashing starts
                cue_timer = core.Clock()
                while cue_timer.getTime() < LETTER_CUE_DUR:
                    if check_escape():
                        running = False
                        break

                    draw_message(win, "Calibration Speller", y=title_y, height=34)
                    draw_grid(
                        cells,
                        target_row=target_row,
                        target_col=target_col,
                        show_target_image_only=True
                    )
                    draw_message(win, f"Focus on letter: {target_letter}", y=bottom_y, height=28)
                    win.flip()
                    core.wait(0.01)

                send_marker(outlet, f"focus_end/{target_letter}")

                if not running:
                    break

                # Row/column flashing
                for seq in range(N_SEQS):
                    send_marker(outlet, f"sequence_start/{target_letter}/{seq}")

                    flash_order = [("row", r) for r in range(GRID_ROWS)] + [("col", c) for c in range(GRID_COLS)]
                    random.shuffle(flash_order)

                    for flash_num, (flash_type, flash_idx) in enumerate(flash_order):
                        if check_escape():
                            running = False
                            break

                        is_target_flash = int(
                            (flash_type == "row" and flash_idx == target_row) or
                            (flash_type == "col" and flash_idx == target_col)
                        )

                        # Flash on
                        draw_message(win, "Calibration Speller", y=title_y, height=34)
                        draw_grid(
                            cells,
                            target_row=target_row,
                            target_col=target_col,
                            highlight_type=flash_type,
                            highlight_index=flash_idx
                        )
                        draw_message(win, f"Focus on letter: {target_letter}", y=bottom_y, height=28)
                        win.flip()

                        send_marker(
                            outlet,
                            f"flash_on/{flash_type}/{flash_idx}/target_{is_target_flash}/char_{target_letter}/seq_{seq}/flash_{flash_num}"
                        )

                        if not wait_with_escape(FLASH_DUR):
                            running = False
                            break

                        send_marker(
                            outlet,
                            f"flash_off/{flash_type}/{flash_idx}/target_{is_target_flash}/char_{target_letter}/seq_{seq}/flash_{flash_num}"
                        )

                        # ISI
                        draw_message(win, "Calibration Speller", y=title_y, height=34)
                        draw_grid(cells)
                        draw_message(win, f"Focus on letter: {target_letter}", y=bottom_y, height=28)
                        win.flip()

                        if not wait_with_escape(ISI):
                            running = False
                            break

                    send_marker(outlet, f"sequence_end/{target_letter}/{seq}")

                    if not running:
                        break

                send_marker(outlet, f"target_end/{target_letter}")

                if not running:
                    break

                # Short break before next target letter
                if letter_idx < len(TARGET_WORD) - 1:
                    if not wait_with_escape(LETTER_BREAK_DUR):
                        running = False
                        break

        # Wait at the end until you close it ---------------------
        while running:
            draw_message(win, "Calibration Speller", y=title_y, height=34)
            draw_grid(cells)
            draw_message(win, "Press ESC or Q to close", y=bottom_y, height=24)
            win.flip()

            if check_escape():
                running = False

            core.wait(0.01)

    finally:
        if outlet is not None and experiment_started:
            try:
                send_marker(outlet, "experiment_end")
            except Exception:
                pass

        if win is not None:
            win.close()
        core.quit()


if __name__ == "__main__":
    main()

Fontconfig warning: ignoring UTF-8: not a valid region tag


Missing image files:
Speller/Images/cell_00.jpg
Speller/Images/cell_01.jpg
Speller/Images/cell_02.jpg
Speller/Images/cell_03.jpg
Speller/Images/cell_04.jpg
Speller/Images/cell_05.jpg
Speller/Images/cell_06.jpg
Speller/Images/cell_07.jpg
Speller/Images/cell_08.jpg
Speller/Images/cell_09.jpg
Speller/Images/cell_10.jpg
Speller/Images/cell_11.jpg
Speller/Images/cell_12.jpg
Speller/Images/cell_13.jpg
Speller/Images/cell_14.jpg
Speller/Images/cell_15.jpg
Speller/Images/cell_16.jpg
Speller/Images/cell_17.jpg
Speller/Images/cell_18.jpg
Speller/Images/cell_19.jpg
Speller/Images/cell_20.jpg
Speller/Images/cell_21.jpg
Speller/Images/cell_22.jpg
Speller/Images/cell_23.jpg
Speller/Images/cell_24.jpg
Speller/Images/cell_25.jpg
Speller/Images/cell_26.jpg
Speller/Images/cell_27.jpg
Speller/Images/cell_28.jpg
Speller/Images/cell_29.jpg
Speller/Images/cell_30.jpg
Speller/Images/cell_31.jpg
Speller/Images/cell_32.jpg
Speller/Images/cell_33.jpg
Speller/Images/cell_34.jpg
Speller/Images/cell_35.jpg
51.9608

SystemExit: 0

/opt/anaconda3/envs/speller/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


: 

# Free spelling (Testing)

Difference with calibration:
- no fixed word
- participants have to mentally choose a letter so:
    - no target 1/0 markers
    - no focus markers 
- predicted text shown on screen

- One sequence = all 6 rows and column flashed once each

In [ ]:
from psychopy import visual, core, event
from PIL import Image
from pylsl import StreamInfo, StreamOutlet, StreamInlet, resolve_byprop, local_clock
import random
import os



# Parameters
# --------------------------------

GRID_ROWS = 6
GRID_COLS = 6

FLASH_DUR = 0.100          # duration of one flash
ISI = 0.075                # pause between flashes
N_SEQS = 10                # number of row/col sequences per letter selection
DECISION_DISPLAY_DUR = 3.0 # time to show decoder output before next flashes start

BG_COLOR = "black"
UNITS = "pix"

X_SPACING = 180
Y_SPACING = 130

CELL_W = 120
CELL_H = 120

IMG_MAX_W = 110
IMG_MAX_H = 110

TEXT_HEIGHT = 36
TEXT_COLOR = "white"
BOX_COLOR = "white"
HIGHLIGHT_COLOR = "yellow"

IMAGE_DIR = "Images"

# Marker outlet sent by this PsychoPy speller
MARKER_STREAM_NAME = "P300Markers_test"
MARKER_STREAM_TYPE = "Markers"

# Decision inlet expected from the online decoder
DECISION_STREAM_NAME = "P300Decisions"

GRID_SYMBOLS = [
    ["A", "B", "C", "D", "E", "F"],
    ["G", "H", "I", "J", "K", "L"],
    ["M", "N", "O", "P", "Q", "R"],
    ["S", "T", "U", "V", "W", "X"],
    ["Y", "Z", "1", "2", "3", "4"],
    ["5", "6", "7", "8", "9", "0"],
]

# Special control symbols you can optionally use
BACKSPACE_SYMBOL = "<"
END_SYMBOL = "_"


# 
# Helper functions
# --------------------------------

def check_escape():
    keys = event.getKeys(keyList=["escape", "q"])
    return "escape" in keys or "q" in keys


def get_image_dir():
    folder = os.path.abspath(IMAGE_DIR)
    if os.path.isdir(folder):
        print("Using image folder:", folder)
        return folder
    raise FileNotFoundError(f"Could not find image folder: {folder}")


def make_image_stim(win, img_path, pos, max_w=IMG_MAX_W, max_h=IMG_MAX_H):
    # Load image size so that we preserve aspect ratio
    with Image.open(img_path) as pil_img:
        orig_w, orig_h = pil_img.size

    scale = min(max_w / orig_w, max_h / orig_h)
    disp_w = max(1, int(orig_w * scale))
    disp_h = max(1, int(orig_h * scale))

    return visual.ImageStim(
        win=win,
        image=img_path,
        pos=pos,
        size=(disp_w, disp_h),
        units=UNITS,
        interpolate=True,
    )


def build_grid_positions(rows, cols, x_spacing, y_spacing):
    positions = []

    total_w = (cols - 1) * x_spacing
    total_h = (rows - 1) * y_spacing

    start_x = -total_w / 2
    start_y = total_h / 2

    for r in range(rows):
        for c in range(cols):
            x = start_x + c * x_spacing
            y = start_y - r * y_spacing
            positions.append((r, c, (x, y)))

    return positions


def draw_message(win, text, y=0, height=28):
    msg = visual.TextStim(
        win=win,
        text=text,
        pos=(0, y),
        height=height,
        color=TEXT_COLOR,
        units=UNITS
    )
    msg.draw()


def wait_with_escape(duration):
    timer = core.Clock()
    while timer.getTime() < duration:
        if check_escape():
            return False
        core.wait(0.005)
    return True


def draw_grid(cells, highlight_type=None, highlight_index=None):
    """
    Normal state:
    show letters in all cells

    During a flash:
    show images in the flashed row or column
    """
    for cell in cells:
        row = cell["row"]
        col = cell["col"]

        is_row_flash = (highlight_type == "row" and row == highlight_index)
        is_col_flash = (highlight_type == "col" and col == highlight_index)
        is_flashed = is_row_flash or is_col_flash

        if is_flashed:
            cell["box"].lineColor = HIGHLIGHT_COLOR
            cell["box"].lineWidth = 4
        else:
            cell["box"].lineColor = BOX_COLOR
            cell["box"].lineWidth = 2

        cell["box"].draw()

        if is_flashed:
            cell["image"].draw()
        else:
            cell["text"].draw()


def make_marker_outlet():
    info = StreamInfo(
        name=MARKER_STREAM_NAME,
        type=MARKER_STREAM_TYPE,
        channel_count=1,
        nominal_srate=0,
        channel_format="string",
        source_id="p300-free-spell-markers"
    )
    return StreamOutlet(info)


def send_marker(outlet, marker):
    outlet.push_sample([marker], local_clock())
    print("[MARKER]", marker)


def connect_decision_inlet(timeout=10):
    """
    Connect to the decoder's output stream.
    The decoder will send one message after each selection,
    for example: decision/A
    """
    print(f"Looking for decoder decision stream '{DECISION_STREAM_NAME}'...")
    streams = resolve_byprop("name", DECISION_STREAM_NAME, timeout=timeout)
    if not streams:
        raise RuntimeError(f"Could not find decision stream '{DECISION_STREAM_NAME}'")
    inlet = StreamInlet(streams[0])
    print("Connected to decoder decision stream.")
    return inlet


def wait_for_decision(inlet, allow_escape=True):
    """
    Wait until the decoder sends a decision marker.
    Expected format:
      decision/A
      decision/B
      decision/_
      decision/<
    """
    while True:
        if allow_escape and check_escape():
            return None

        sample, ts = inlet.pull_sample(timeout=0.01)
        if ts is None:
            continue

        msg = sample[0]
        print("[DECISION]", msg)

        if msg.startswith("decision/"):
            parts = msg.split("/", 1)
            if len(parts) == 2:
                return parts[1]


def update_text(current_text, predicted_char):
    """
    Optional text control logic.
    BACKSPACE_SYMBOL removes one char.
    END_SYMBOL inserts a space.
    """
    if predicted_char == BACKSPACE_SYMBOL:
        return current_text[:-1] if len(current_text) > 0 else current_text
    elif predicted_char == END_SYMBOL:
        return current_text + " "
    else:
        return current_text + predicted_char


# 
# Main
# --------------------------------

def main():
    win = None
    marker_outlet = None
    decision_inlet = None
    experiment_started = False

    try:
        image_dir_abs = get_image_dir()
        marker_outlet = make_marker_outlet()
        decision_inlet = connect_decision_inlet(timeout=15)

        win = visual.Window(
            fullscr=True,
            color=BG_COLOR,
            units=UNITS,
            checkTiming=False
        )

        win_w, win_h = win.size
        title_y = win_h / 2 - 60
        bottom_y = -win_h / 2 + 50
        typed_y = title_y - 50
        grid_center_shift_y = -40

        # Build image paths
        image_files = [f"cell_{i:02d}.jpg" for i in range(GRID_ROWS * GRID_COLS)]
        image_paths = [os.path.join(image_dir_abs, f) for f in image_files]

        missing = [p for p in image_paths if not os.path.exists(p)]
        if missing:
            print("Missing image files:")
            for m in missing:
                print(m)
            return

        # Build cells
        cells = []
        positions = build_grid_positions(GRID_ROWS, GRID_COLS, X_SPACING, Y_SPACING)

        for idx, (row, col, pos) in enumerate(positions):
            symbol = GRID_SYMBOLS[row][col]
            shifted_pos = (pos[0], pos[1] + grid_center_shift_y)

            box_stim = visual.Rect(
                win=win,
                pos=shifted_pos,
                width=CELL_W,
                height=CELL_H,
                units=UNITS,
                lineColor=BOX_COLOR,
                fillColor=None,
                lineWidth=2
            )

            text_stim = visual.TextStim(
                win=win,
                text=symbol,
                pos=shifted_pos,
                height=TEXT_HEIGHT,
                color=TEXT_COLOR,
                units=UNITS
            )

            image_stim = make_image_stim(
                win=win,
                img_path=image_paths[idx],
                pos=shifted_pos,
                max_w=IMG_MAX_W,
                max_h=IMG_MAX_H
            )

            cells.append({
                "index": idx,
                "row": row,
                "col": col,
                "symbol": symbol,
                "pos": shifted_pos,
                "box": box_stim,
                "text": text_stim,
                "image": image_stim
            })

        typed_text = ""
        selection_idx = 0
        running = True

        # Instruction screen
        while running:
            if check_escape():
                running = False
                break

            draw_message(win, "Free Spelling P300", y=title_y, height=34)
            draw_message(win, f"Typed: {typed_text}", y=typed_y, height=28)
            draw_grid(cells)
            draw_message(win, "Press SPACE to start, ESC or Q to quit", y=bottom_y, height=24)
            win.flip()

            event.clearEvents(eventType="keyboard")
            keys = event.waitKeys(keyList=["space", "escape", "q"])

            if keys:
                if "space" in keys:
                    send_marker(marker_outlet, "experiment_start")
                    experiment_started = True
                    break
                if "escape" in keys or "q" in keys:
                    running = False
                    break

        # Main free-spelling loop
        while running:
            if check_escape():
                running = False
                break

            # Short prompt before this character selection begins
            prompt_timer = core.Clock()
            while prompt_timer.getTime() < 1.0:
                if check_escape():
                    running = False
                    break

                draw_message(win, "Free Spelling P300", y=title_y, height=34)
                draw_message(win, f"Typed: {typed_text}", y=typed_y, height=28)
                draw_grid(cells)
                draw_message(win, "Focus on the character you want to select", y=bottom_y, height=24)
                win.flip()
                core.wait(0.01)

            if not running:
                break

            send_marker(marker_outlet, f"selection_start/{selection_idx}")

            # Run N_SEQS row/column sequences
            for seq in range(N_SEQS):
                send_marker(marker_outlet, f"sequence_start/{selection_idx}/{seq}")

                flash_order = [("row", r) for r in range(GRID_ROWS)] + [("col", c) for c in range(GRID_COLS)]
                random.shuffle(flash_order)

                for flash_num, (flash_type, flash_idx) in enumerate(flash_order):
                    if check_escape():
                        running = False
                        break

                    # Draw flashed grid
                    draw_message(win, "Free Spelling P300", y=title_y, height=34)
                    draw_message(win, f"Typed: {typed_text}", y=typed_y, height=28)
                    draw_grid(cells, highlight_type=flash_type, highlight_index=flash_idx)
                    draw_message(win, "Focus on your chosen character", y=bottom_y, height=24)
                    win.flip()

                    # flash_on marker
                    # No target label during testing
                    send_marker(
                        marker_outlet,
                        f"flash_on/{flash_type}/{flash_idx}/sel_{selection_idx}/seq_{seq}/flash_{flash_num}"
                    )

                    if not wait_with_escape(FLASH_DUR):
                        running = False
                        break

                    # flash_off marker
                    send_marker(
                        marker_outlet,
                        f"flash_off/{flash_type}/{flash_idx}/sel_{selection_idx}/seq_{seq}/flash_{flash_num}"
                    )

                    # ISI screen
                    draw_message(win, "Free Spelling P300", y=title_y, height=34)
                    draw_message(win, f"Typed: {typed_text}", y=typed_y, height=28)
                    draw_grid(cells)
                    draw_message(win, "Focus on your chosen character", y=bottom_y, height=24)
                    win.flip()

                    if not wait_with_escape(ISI):
                        running = False
                        break

                send_marker(marker_outlet, f"sequence_end/{selection_idx}/{seq}")

                if not running:
                    break

            if not running:
                break

            send_marker(marker_outlet, f"selection_end/{selection_idx}")

            # Wait for decoder decision
            draw_message(win, "Free Spelling P300", y=title_y, height=34)
            draw_message(win, f"Typed: {typed_text}", y=typed_y, height=28)
            draw_grid(cells)
            draw_message(win, "Waiting for decoder...", y=bottom_y, height=24)
            win.flip()

            predicted_char = wait_for_decision(decision_inlet, allow_escape=True)
            if predicted_char is None:
                running = False
                break

            typed_text = update_text(typed_text, predicted_char)

            # Show decoder output for 3 seconds before new flashes start
            decision_timer = core.Clock()
            while decision_timer.getTime() < DECISION_DISPLAY_DUR:
                if check_escape():
                    running = False
                    break

                draw_message(win, "Free Spelling P300", y=title_y, height=34)
                draw_message(win, f"Typed: {typed_text}", y=typed_y, height=28)
                draw_grid(cells)
                draw_message(win, f"Decoder chose: {predicted_char}", y=bottom_y, height=24)
                win.flip()
                core.wait(0.01)

            if not running:
                break

            selection_idx += 1

        # End screen
        while running:
            draw_message(win, "Free Spelling P300", y=title_y, height=34)
            draw_message(win, f"Final text: {typed_text}", y=typed_y, height=28)
            draw_grid(cells)
            draw_message(win, "Press ESC or Q to close", y=bottom_y, height=24)
            win.flip()

            if check_escape():
                running = False

            core.wait(0.01)

    finally:
        if marker_outlet is not None and experiment_started:
            try:
                send_marker(marker_outlet, "experiment_end")
            except Exception:
                pass

        if win is not None:
            win.close()
        core.quit()


if __name__ == "__main__":
    main()